In [1]:
1+1

2

In [2]:
import pandas as pd
df=pd.read_csv("../AQI_Data/processed_aqi_data.csv")
df.head()

,timestamp,location_name,location_lat,location_lon,co,no2,o3,pm10,pm25,so2,aqi,aqi_category,hour,day,month,year,day_of_week,is_weekend
0,2019-01-01 00:00:00,Delhi,28.7041,77.1025,0.598089,24.943655,18.081507,108.281832,75.196148,4.378213,221.629670,Poor,0,1,1,2019,1,0
1,2019-01-01 00:00:00,Faridabad,28.4089,77.3178,1.093698,5.699821,53.686134,56.260069,42.709284,6.493686,137.180953,Moderate,0,1,1,2019,1,0
2,2019-01-01 00:00:00,Ghaziabad,28.6692,77.4538,0.420770,19.419563,34.967340,82.842934,59.295449,4.423260,205.059468,Poor,0,1,1,2019,1,0
3,2019-01-01 00:00:00,Gurgaon,28.4595,77.0266,0.706815,30.585009,35.874847,117.015282,75.435553,7.511787,221.879156,Poor,0,1,1,2019,1,0
4,2019-01-01 00:00:00,Noida,28.5355,77.3910,0.542118,23.626271,39.401078,116.208836,68.843636,6.828320,215.009684,Poor,0,1,1,2019,1,0


In [3]:
df.tail()

,timestamp,location_name,location_lat,location_lon,co,no2,o3,pm10,pm25,so2,aqi,aqi_category,hour,day,month,year,day_of_week,is_weekend
219120,2024-01-01 00:00:00,Delhi,28.7041,77.1025,1.061767,18.940857,30.316565,114.754317,88.277907,6.874081,235.262240,Poor,0,1,1,2024,0,0
219121,2024-01-01 00:00:00,Faridabad,28.4089,77.3178,0.000000,16.321801,51.374857,105.162561,73.894611,4.971524,220.273332,Poor,0,1,1,2024,0,0
219122,2024-01-01 00:00:00,Ghaziabad,28.6692,77.4538,0.585862,32.556168,30.104965,40.884687,39.172260,6.823918,119.672685,Moderate,0,1,1,2024,0,0
219123,2024-01-01 00:00:00,Gurgaon,28.4595,77.0266,0.662425,13.672529,18.609753,103.387562,69.809989,9.142254,216.016726,Poor,0,1,1,2024,0,0
219124,2024-01-01 00:00:00,Noida,28.5355,77.3910,0.326056,27.443304,46.079022,94.015069,41.227786,4.909004,129.847540,Moderate,0,1,1,2024,0,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 219125 entries, 0 to 219124
Data columns (total 18 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   timestamp      219125 non-null  str    
 1   location_name  219125 non-null  str    
 2   location_lat   219125 non-null  float64
 3   location_lon   219125 non-null  float64
 4   co             219125 non-null  float64
 5   no2            219125 non-null  float64
 6   o3             219125 non-null  float64
 7   pm10           219125 non-null  float64
 8   pm25           219125 non-null  float64
 9   so2            219125 non-null  float64
 10  aqi            219125 non-null  float64
 11  aqi_category   219125 non-null  str    
 12  hour           219125 non-null  int64  
 13  day            219125 non-null  int64  
 14  month          219125 non-null  int64  
 15  year           219125 non-null  int64  
 16  day_of_week    219125 non-null  int64  
 17  is_weekend     219125 non-null  int64  


In [5]:
df.shape

(219125, 18)

In [6]:
df.nunique()

timestamp         43825
location_name         5
location_lat          5
location_lon          5
co               210386
no2              205013
o3               211649
pm10             212358
pm25             213627
so2              210216
aqi              213627
aqi_category          5
hour                 24
day                  31
month                12
year                  6
day_of_week           7
is_weekend            2
dtype: int64

In [7]:
df['location_name'].isnull().sum()

np.int64(0)

In [8]:
df.isnull().sum()

timestamp        0
location_name    0
location_lat     0
location_lon     0
co               0
no2              0
o3               0
pm10             0
pm25             0
so2              0
aqi              0
aqi_category     0
hour             0
day              0
month            0
year             0
day_of_week      0
is_weekend       0
dtype: int64

In [9]:
df.drop("aqi_category",axis=1,inplace=True)

In [10]:
df.shape

(219125, 17)

AQI forecasting mein timestamp ko sort/check karne ke baad drop karna hai. Direct starting mein drop mat karna, kyunki hume pehle verify karna hai ki hourly data properly ordered hai.

In [11]:
# Har location ke liye alag-alag (groupby zaroori hai, warna locations mix ho jaayenge)
df = df.sort_values(['location_name', 'timestamp'])  # timestamp abhi bhi rakha hai na?

for col in ['pm25', 'pm10', 'aqi']:
    df[f'{col}_lag_1hr'] = df.groupby('location_name')[col].shift(1)
    df[f'{col}_lag_3hr'] = df.groupby('location_name')[col].shift(3)
    df[f'{col}_rolling_mean_6hr'] = df.groupby('location_name')[col].transform(lambda x: x.rolling(6).mean())

In [12]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

In [13]:
df.dropna(subset=['timestamp'], inplace=True)  ## not become nan after convert str->datetime

In [14]:
df.shape

(219125, 26)

In [15]:
df.head()

,timestamp,location_name,location_lat,location_lon,co,no2,o3,pm10,pm25,so2,...,is_weekend,pm25_lag_1hr,pm25_lag_3hr,pm25_rolling_mean_6hr,pm10_lag_1hr,pm10_lag_3hr,pm10_rolling_mean_6hr,aqi_lag_1hr,aqi_lag_3hr,aqi_rolling_mean_6hr
0,2019-01-01 00:00:00,Delhi,28.7041,77.1025,0.598089,24.943655,18.081507,108.281832,75.196148,4.378213,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2019-01-01 01:00:00,Delhi,28.7041,77.1025,1.040468,21.877824,55.547468,110.226722,65.451836,5.780110,...,0,75.196148,NaN,NaN,108.281832,NaN,NaN,221.629670,NaN,NaN
10,2019-01-01 02:00:00,Delhi,28.7041,77.1025,0.435431,24.014983,19.364570,44.680995,24.225843,4.823612,...,0,65.451836,NaN,NaN,110.226722,NaN,NaN,211.475071,NaN,NaN
15,2019-01-01 03:00:00,Delhi,28.7041,77.1025,0.641405,46.650006,49.052488,106.661734,65.056997,6.407425,...,0,24.225843,75.196148,NaN,44.680995,108.281832,NaN,76.601125,221.629670,NaN
20,2019-01-01 04:00:00,Delhi,28.7041,77.1025,0.634266,45.782184,47.741928,113.304567,70.506701,9.329313,...,0,65.056997,65.451836,NaN,106.661734,110.226722,NaN,211.063608,211.475071,NaN


In [16]:
df.isnull().sum()

timestamp                 0
location_name             0
location_lat              0
location_lon              0
co                        0
no2                       0
o3                        0
pm10                      0
pm25                      0
so2                       0
aqi                       0
hour                      0
day                       0
month                     0
year                      0
day_of_week               0
is_weekend                0
pm25_lag_1hr              5
pm25_lag_3hr             15
pm25_rolling_mean_6hr    25
pm10_lag_1hr              5
pm10_lag_3hr             15
pm10_rolling_mean_6hr    25
aqi_lag_1hr               5
aqi_lag_3hr              15
aqi_rolling_mean_6hr     25
dtype: int64

In [17]:
df.shape

(219125, 26)

In [18]:
lag_cols = ['pm25_lag_1hr', 'pm25_lag_3hr', 'pm25_rolling_mean_6hr',
            'pm10_lag_1hr', 'pm10_lag_3hr', 'pm10_rolling_mean_6hr',
            'aqi_lag_1hr', 'aqi_lag_3hr', 'aqi_rolling_mean_6hr']

df.dropna(subset=lag_cols, inplace=True)
df.reset_index(drop=True, inplace=True)

print(df.shape)  

(219100, 26)


In [19]:
print(df.select_dtypes(include="number").columns.tolist())

['location_lat', 'location_lon', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2', 'aqi', 'hour', 'day', 'month', 'year', 'day_of_week', 'is_weekend', 'pm25_lag_1hr', 'pm25_lag_3hr', 'pm25_rolling_mean_6hr', 'pm10_lag_1hr', 'pm10_lag_3hr', 'pm10_rolling_mean_6hr', 'aqi_lag_1hr', 'aqi_lag_3hr', 'aqi_rolling_mean_6hr']


In [20]:
df['location_name'].unique()

<StringArray>
['Delhi', 'Faridabad', 'Ghaziabad', 'Gurgaon', 'Noida']
Length: 5, dtype: str

In [21]:
from sklearn.preprocessing import OneHotEncoder
OneHot_encoder=OneHotEncoder(sparse_output=False)
location_name_encoded=OneHot_encoder.fit_transform(df[['location_name']])

In [22]:
location_name_encoded_df=pd.DataFrame(location_name_encoded,columns=OneHot_encoder.get_feature_names_out(['location_name']))
location_name_encoded_df

,location_name_Delhi,location_name_Faridabad,location_name_Ghaziabad,location_name_Gurgaon,location_name_Noida
0,1.0,0.0,0.0,0.0,0.0
1,1.0,0.0,0.0,0.0,0.0
2,1.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
219095,0.0,0.0,0.0,0.0,1.0
219096,0.0,0.0,0.0,0.0,1.0
219097,0.0,0.0,0.0,0.0,1.0
219098,0.0,0.0,0.0,0.0,1.0


In [23]:
df=pd.concat([df.drop("location_name",axis=1),location_name_encoded_df],axis=1)

In [24]:
df.head()


,timestamp,location_lat,location_lon,co,no2,o3,pm10,pm25,so2,aqi,...,pm10_lag_3hr,pm10_rolling_mean_6hr,aqi_lag_1hr,aqi_lag_3hr,aqi_rolling_mean_6hr,location_name_Delhi,location_name_Faridabad,location_name_Ghaziabad,location_name_Gurgaon,location_name_Noida
0,2019-01-01 05:00:00,28.7041,77.1025,0.816863,41.197202,62.702559,140.558978,86.095715,6.883921,232.988166,...,44.680995,103.952471,216.742773,76.601125,195.083402,1.0,0.0,0.0,0.0,0.0
1,2019-01-01 06:00:00,28.7041,77.1025,0.929340,34.550916,51.396836,99.014741,66.606326,8.540332,212.678172,...,106.661734,102.407956,232.988166,211.063608,193.591486,1.0,0.0,0.0,0.0,0.0
2,2019-01-01 07:00:00,28.7041,77.1025,1.052251,33.843076,52.311432,112.633437,89.074494,2.942878,236.092368,...,113.304567,102.809075,212.678172,216.742773,197.694369,1.0,0.0,0.0,0.0,0.0
3,2019-01-01 08:00:00,28.7041,77.1025,0.609387,30.493068,41.554033,151.415465,100.624724,2.679027,248.128923,...,140.558978,120.598154,236.092368,232.988166,226.282335,1.0,0.0,0.0,0.0,0.0
4,2019-01-01 09:00:00,28.7041,77.1025,0.496634,38.361836,34.762022,78.789606,56.916829,4.339448,202.580696,...,99.014741,115.952799,248.128923,212.678172,224.868516,1.0,0.0,0.0,0.0,0.0


In [25]:
df.shape

(219100, 30)

In [26]:
df = df.drop(columns=["timestamp"])

In [27]:
df.to_csv('../AQI_Data/AQI_cleaned.csv', index=False)

In [28]:
df.duplicated().sum()

np.int64(0)

In [29]:
print(df.columns.tolist())

['location_lat', 'location_lon', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2', 'aqi', 'hour', 'day', 'month', 'year', 'day_of_week', 'is_weekend', 'pm25_lag_1hr', 'pm25_lag_3hr', 'pm25_rolling_mean_6hr', 'pm10_lag_1hr', 'pm10_lag_3hr', 'pm10_rolling_mean_6hr', 'aqi_lag_1hr', 'aqi_lag_3hr', 'aqi_rolling_mean_6hr', 'location_name_Delhi', 'location_name_Faridabad', 'location_name_Ghaziabad', 'location_name_Gurgaon', 'location_name_Noida']
